In [1]:
BASE_PATH = '/mnt/Toshiba/Code/intern_ml_test_task/task'

In [2]:
import os

LABELS_PATH = os.path.join(BASE_PATH, "data/train_labels.jsonl")
PREDS_PATH = os.path.join(BASE_PATH, "data/train_predictions.jsonl")

In [3]:
import json
import re
import sys
from difflib import SequenceMatcher

In [4]:
def normalize(text: str) -> str:
    """Uppercase, collapse whitespace, strip trailing dots/spaces for robust compare."""
    text = text.upper().strip()
    text = re.sub(r"\s+", " ", text)
    return text

def extract_predicted_text(viz_entry: dict) -> str:
    """Join the OCR line texts of one field into the predicted string."""
    lines = viz_entry.get("lines", [])
    return " ".join(l.get("text", "") for l in lines).strip()

In [5]:
# Preanalysis of failed (non-exact) fields:
#  1. max char_conf at which OCR still fails (per-char confidence of a wrong char)
#  2. max number of char errors inside one failed field (and one failed line)
#  3. which GT chars fail the most (+ top confusion pairs)

from collections import Counter


def pred_string_with_conf(viz_entry):
    """Return (pred_str, confs) aligned per char; inter-line separator spaces get conf=None."""
    chars, confs = [], []
    for j, line in enumerate(viz_entry.get("lines", [])):
        if j:
            chars.append(" ")
            confs.append(None)
        text = line.get("text", "")
        cc = line.get("char_conf", [])
        for k, ch in enumerate(text):
            chars.append(ch)
            confs.append(cc[k] if k < len(cc) else None)
    return "".join(chars), confs


fail_conf = []                # char_conf of each wrong predicted char
fail_gt_chars = Counter()     # how often each GT char is mis-recognized
fail_pairs = Counter()        # (gt_char, pred_char) confusion pairs
max_errors_field = (0, None)  # (n_errors, (doc_id, field))
max_errors_line = (0, None)   # (n_errors, doc_id)
n_total_lines = n_failed_lines = n_failed_fields = 0

with open(LABELS_PATH, encoding="utf-8") as f_labels, open(PREDS_PATH, encoding="utf-8") as f_preds:
    for labels_line, preds_line in zip(f_labels, f_preds):
        label_rec, pred_rec = json.loads(labels_line), json.loads(preds_line)
        gt_fields = label_rec["fields"]
        viz_fields = pred_rec["viz"]
        n_total_lines += 1

        line_errors = 0
        line_failed = False
        for field, gt in gt_fields.items():
            viz = viz_fields.get(field)
            if viz is None:
                continue
            pred, confs = pred_string_with_conf(viz)
            if gt == pred:
                continue
            line_failed = True
            n_failed_fields += 1

            errors = 0
            sm = SequenceMatcher(None, gt, pred, autojunk=False)
            for tag, i1, i2, j1, j2 in sm.get_opcodes():
                if tag == "equal":
                    continue
                for k in range(max(i2 - i1, j2 - j1)):
                    gt_ch = gt[i1 + k] if i1 + k < i2 else None
                    pred_ch = pred[j1 + k] if j1 + k < j2 else None
                    if gt_ch == pred_ch:
                        continue
                    errors += 1
                    if gt_ch is not None:
                        fail_gt_chars[gt_ch] += 1
                        fail_pairs[(gt_ch, pred_ch)] += 1
                    else:  # pure insertion: predicted char that should not be there
                        fail_pairs[("<none>", pred_ch)] += 1
                    if pred_ch is not None and confs[j1 + k] is not None:
                        fail_conf.append(confs[j1 + k])

            line_errors += errors
            if errors > max_errors_field[0]:
                max_errors_field = (errors, (label_rec["doc_id"], field))

        if line_failed:
            n_failed_lines += 1
        if line_errors > max_errors_line[0]:
            max_errors_line = (line_errors, label_rec["doc_id"])

print(f"total lines: {n_total_lines}, failed lines: {n_failed_lines}, failed fields: {n_failed_fields}")
print()
print(f"1) Max char_conf with which OCR still fails: {max(fail_conf):.4f}")
print(f"   wrong chars with conf >= 0.99: {sum(c >= 0.99 for c in fail_conf)},"
      f" >= 0.999: {sum(c >= 0.999 for c in fail_conf)} (out of {len(fail_conf)} wrong chars with known conf)")
print()
print(f"2) Max char errors in one failed field: {max_errors_field[0]}  at {max_errors_field[1]}")
print(f"   Max char errors in one failed line:  {max_errors_line[0]}  at doc {max_errors_line[1]}")
print()
print("3) Most failing GT chars (top 15):")
for ch, n in fail_gt_chars.most_common(15):
    print(f"   {ch!r}: {n}")
print()
print("Top confusion pairs (gt -> pred, top 15):")
for (gt_ch, pred_ch), n in fail_pairs.most_common(15):
    print(f"   {gt_ch!r} -> {pred_ch!r}: {n}")

total lines: 3000, failed lines: 1217, failed fields: 1838

1) Max char_conf with which OCR still fails: 1.0000
   wrong chars with conf >= 0.99: 247, >= 0.999: 126 (out of 1803 wrong chars with known conf)

2) Max char errors in one failed field: 53  at ('tr_00104', 'Issue_organization_ru')
   Max char errors in one failed line:  57  at doc tr_00104

3) Most failing GT chars (top 15):
   'О': 380
   ' ': 306
   'С': 245
   'А': 236
   'Р': 199
   '.': 185
   'Н': 177
   'К': 177
   'И': 164
   'Е': 151
   'В': 131
   'Л': 113
   'М': 109
   '0': 96
   'Т': 93

Top confusion pairs (gt -> pred, top 15):
   ' ' -> None: 295
   'О' -> None: 218
   '.' -> None: 180
   'О' -> 'С': 143
   'С' -> None: 137
   'А' -> None: 125
   'Р' -> None: 113
   'К' -> None: 110
   'А' -> 'Л': 102
   'С' -> 'О': 95
   'Н' -> None: 95
   'Е' -> None: 90
   'И' -> None: 85
   'Р' -> 'В': 81
   'В' -> None: 77


In [6]:
# Correlation analysis: are OCR failures predictable from image quality?
# Per line: total char errors (missing viz fields count as len(gt) missing chars),
# failed flag (line is not 100% correct) vs quality metrics (glare, sharpness, doc_px).

from statistics import correlation


def count_line_errors(gt_fields, viz_fields):
    """Total char errors in a line; a field without prediction contributes all its chars."""
    total = 0
    for field, gt in gt_fields.items():
        viz = viz_fields.get(field)
        if viz is None:
            total += len(gt)
            continue
        pred, _ = pred_string_with_conf(viz)
        if gt == pred:
            continue
        sm = SequenceMatcher(None, gt, pred, autojunk=False)
        for tag, i1, i2, j1, j2 in sm.get_opcodes():
            if tag != "equal":
                total += max(i2 - i1, j2 - j1)
    return total


rows = []  # (errors, failed, glare, sharpness, doc_px, source)
with open(LABELS_PATH, encoding="utf-8") as f_labels, open(PREDS_PATH, encoding="utf-8") as f_preds:
    for labels_line, preds_line in zip(f_labels, f_preds):
        label_rec, pred_rec = json.loads(labels_line), json.loads(preds_line)
        errors = count_line_errors(label_rec["fields"], pred_rec["viz"])
        q = pred_rec.get("quality", {})
        rows.append((errors, errors > 0, q.get("glare"), q.get("sharpness"),
                     q.get("doc_px"), pred_rec.get("source")))

errors_col = [r[0] for r in rows]
failed_col = [float(r[1]) for r in rows]

print("Pearson correlation with char errors per line:")
for name, col in (("glare", 2), ("sharpness", 3), ("doc_px", 4)):
    vals = [r[col] for r in rows]
    print(f"   {name:<10} r = {correlation(vals, errors_col):+.3f}")
print("Pearson correlation with failed-line flag:")
for name, col in (("glare", 2), ("sharpness", 3), ("doc_px", 4)):
    vals = [r[col] for r in rows]
    print(f"   {name:<10} r = {correlation(vals, failed_col):+.3f}")
print()


def bucket_table(col_idx, bins, labels):
    print(f"   by {['glare', 'sharpness', 'doc_px'][col_idx - 2]} (bins: {labels}):")
    print(f"   {'bin':<18}{'lines':>7}{'mean errors':>13}{'% failed':>10}")
    for lo, hi, lab in zip(bins[:-1], bins[1:], labels):
        sel = [r for r in rows if lo <= r[col_idx] < hi]
        if not sel:
            continue
        mean_err = sum(r[0] for r in sel) / len(sel)
        pct_fail = 100 * sum(r[1] for r in sel) / len(sel)
        print(f"   {lab:<18}{len(sel):>7}{mean_err:>13.2f}{pct_fail:>9.1f}%")
    print()


print("Bucketed quality analysis:")
bucket_table(2, [0, .05, .10, .15, .20, .25, 1.0],
             ["glare<0.05", "0.05-0.10", "0.10-0.15", "0.15-0.20", "0.20-0.25", "0.25+"])
bucket_table(3, [0, 75, 100, 125, 150, 200, 10_000],
             ["sharp<75", "75-100", "100-125", "125-150", "150-200", "200+"])
bucket_table(4, [0, 700, 1000, 1500, 2500, 10**9],
             ["px<700", "700-1000", "1000-1500", "1500-2500", "2500+"])

print("By source:")
print(f"   {'source':<10}{'lines':>7}{'mean errors':>13}{'% failed':>10}")
for src in sorted({r[5] for r in rows}):
    sel = [r for r in rows if r[5] == src]
    mean_err = sum(r[0] for r in sel) / len(sel)
    pct_fail = 100 * sum(r[1] for r in sel) / len(sel)
    print(f"   {src:<10}{len(sel):>7}{mean_err:>13.2f}{pct_fail:>9.1f}%")

Pearson correlation with char errors per line:
   glare      r = +0.197
   sharpness  r = -0.190
   doc_px     r = -0.194
Pearson correlation with failed-line flag:
   glare      r = +0.366
   sharpness  r = -0.384
   doc_px     r = -0.384

Bucketed quality analysis:
   by glare (bins: ['glare<0.05', '0.05-0.10', '0.10-0.15', '0.15-0.20', '0.20-0.25', '0.25+']):
   bin                 lines  mean errors  % failed
   glare<0.05           1582         1.63     33.5%
   0.05-0.10             756         2.19     39.8%
   0.10-0.15             328         3.47     65.5%
   0.15-0.20             188         6.94     83.0%
   0.20-0.25             104         6.21     98.1%
   0.25+                  42         8.02     97.6%

   by sharpness (bins: ['sharp<75', '75-100', '100-125', '125-150', '150-200', '200+']):
   bin                 lines  mean errors  % failed
   sharp<75               19         7.37    100.0%
   75-100                 94         6.89     97.9%
   100-125               

- __Корреляция умеренная (r ≈ ±0.37), но таблица по корзинам почти ступенчатая.__ Линейная корреляция Пирсона выглядит скромно, однако если разбить данные на интервалы, картина резкая:

  - glare < 0.05 → отказ ~34% строк
  - glare 0.10–0.15 → ~65%
  - glare ≥ 0.20 → ~98%

  То есть зависимость не «чем больше glare, тем пропорционально больше ошибок», а «до некоторого порока ошибки редки, после порога — почти гарантированы».

- __Практический вывод:__ не нужно сложное предсказание количества ошибок — достаточно простого правила-порога (quality gate). Например: `glare > 0.15` или `sharpness < 125` ⇒ пометить строку как ненадёжную. Такое бинарное правило «отсечёт» большинство проблемных строк, почти не затронув хорошие.

- __Что при этом не предсказывается порогом:__ *число* ошибок внутри отказавшей строки (корреляция всего |r| ≈ 0.19, распределение тяжёлое — бывают строки с 1 ошибкой, бывают с 57). Качество говорит __попадёт ли строка в отказ__, но не __насколько сильно она сломается__.


In [7]:
# Char-conf based auto-reject, step 1: which threshold is safe?
# (a) distribution of char conf for WRONG chars vs CORRECT chars
# (b) char-level sweep: % of wrong chars and % of ok chars below threshold T
# (c) line-level confusion matrix for rule "reject line if any char conf < T"

wrong_confs = []      # conf of actually-wrong predicted chars
ok_confs = []         # conf of correctly predicted chars
line_stats = []       # (failed, min_conf_or_None) per line
n_lines = 0

with open(LABELS_PATH, encoding="utf-8") as f_labels, open(PREDS_PATH, encoding="utf-8") as f_preds:
    for labels_line, preds_line in zip(f_labels, f_preds):
        label_rec, pred_rec = json.loads(labels_line), json.loads(preds_line)
        gt_fields, viz_fields = label_rec["fields"], pred_rec["viz"]
        n_lines += 1
        failed = count_line_errors(gt_fields, viz_fields) > 0
        min_conf = None
        for field, gt in gt_fields.items():
            viz = viz_fields.get(field)
            if viz is None:
                continue  # missing field: no chars/conf at all (failed already counted)
            pred, confs = pred_string_with_conf(viz)
            sm = SequenceMatcher(None, gt, pred, autojunk=False)
            for tag, i1, i2, j1, j2 in sm.get_opcodes():
                if tag == "equal":
                    for k in range(i2 - i1):
                        c = confs[j1 + k]
                        if c is not None:
                            ok_confs.append(c)
                else:
                    for k in range(max(i2 - i1, j2 - j1)):
                        gt_ch = gt[i1 + k] if i1 + k < i2 else None
                        pred_ch = pred[j1 + k] if j1 + k < j2 else None
                        if gt_ch == pred_ch:
                            continue
                        if pred_ch is not None and confs[j1 + k] is not None:
                            wrong_confs.append(confs[j1 + k])
            for c in confs:
                if c is not None and (min_conf is None or c < min_conf):
                    min_conf = c
        line_stats.append((failed, min_conf))


def dist_stats(vals, name):
    s = sorted(vals)
    n = len(s)
    mean = sum(s) / n
    def p(q): return s[min(n - 1, int(q * n))]
    print(f"{name}: n={n}, mean={mean:.4f}, "
          f"p5={p(.05):.4f}, p25={p(.25):.4f}, median={p(.5):.4f}, "
          f"p75={p(.75):.4f}, p95={p(.95):.4f}")


print("Distribution of char conf:")
dist_stats(wrong_confs, "  WRONG chars ")
dist_stats(ok_confs, "  OK chars    ")
print()

THRESHOLDS = (0.90, 0.92, 0.94, 0.95, 0.96, 0.97, 0.98, 0.99, 0.995, 0.999)
print("Char-level threshold sweep:")
print(f"{'T':>8}{'wrong<T (catch)':>17}{'ok<T (collateral)':>19}")
for t in THRESHOLDS:
    w = 100 * sum(c < t for c in wrong_confs) / len(wrong_confs)
    o = 100 * sum(c < t for c in ok_confs) / len(ok_confs)
    print(f"{t:>8}{w:>16.1f}%{o:>18.1f}%")
print()

print("Line-level confusion matrix for rule 'reject line if any char conf < T':")
print(f"{'T':>8}{'recall %':>10}{'false alarm %':>15}{'approved-fail %':>17}")
for t in THRESHOLDS:
    ff = fo = af = ao = 0
    for failed, min_conf in line_stats:
        predicted_failed = min_conf is not None and min_conf < t
        if predicted_failed and failed:
            ff += 1
        elif predicted_failed and not failed:
            fo += 1
        elif not predicted_failed and failed:
            af += 1
        else:
            ao += 1
    recall = 100 * ff / (ff + af)
    falar = 100 * fo / (fo + ao)
    danger = 100 * af / len(line_stats)
    print(f"{t:>8}{recall:>10.1f}{falar:>15.1f}{danger:>17.2f}")

Distribution of char conf:
  WRONG chars : n=1803, mean=0.6597, p5=0.2524, p25=0.4473, median=0.6695, p75=0.8770, p95=0.9995
  OK chars    : n=470547, mean=0.9961, p5=0.9862, p25=0.9982, median=0.9995, p75=0.9999, p95=1.0000

Char-level threshold sweep:
       T  wrong<T (catch)  ok<T (collateral)
     0.9            78.1%               0.6%
    0.92            80.3%               0.8%
    0.94            81.9%               1.1%
    0.95            82.3%               1.3%
    0.96            82.6%               1.7%
    0.97            83.4%               2.4%
    0.98            84.7%               3.6%
    0.99            86.3%               6.6%
   0.995            88.2%              11.6%
   0.999            93.0%              35.0%

Line-level confusion matrix for rule 'reject line if any char conf < T':
       T  recall %  false alarm %  approved-fail %
     0.9      75.5            2.9            11.00
    0.92      77.7            4.2            10.00
    0.94      79.6      

In [8]:
# Cascade checker: reject the line if ANY rule triggers:
#   R1: unknown Issue_organisation_code (reference)
#   R2: Issue_organization_ru not similar to any alias of the predicted code
#       (fuzzy: best SequenceMatcher ratio vs the code's reference names
#       < R2_SIM_T; exact match is useless here because the reference keeps
#       old instrumental-case/abbreviated names, e.g. 'ОТДЕЛЕНИЕМ УФМС ...
#       РЕСП. ... РАЙОНЕ' vs the labels' 'ОТДЕЛ ПО ВОПРОСАМ МИГРАЦИИ ОМВД
#       ... РЕСПУБЛИКЕ ... Р-НЕ' — only 6.2% of GT names are in the reference)
#   C : any predicted char with conf < CHAR_CONF_T
#   M : some gt field has no prediction (viz value is None)
#   Q : quality gate glare > GLARE_MAX or sharpness < SHARPNESS_MIN
#   F : predicted value violates its field format (see FMT_* below):
#       no digits in ФИО, Sex_ru == 'МУЖ.'/'ЖЕН.', dates == dd.mm.yyyy,
#       charset checks for code / licence / birth place / organisation.
#       NOTE: the TEXT charset also allows ( ) , № / ' " ’ and Latin
#       letters — GT values legitimately contain them (e.g. 'САХА
#       (ЯКУТИЯ)', '№', foreign birth places like 'Г. SVERDLOVS'K'), so a
#       stricter charset would flag correct predictions (136 on train).
# Same confusion matrix; per-rule contributions show what adds what.
#
# 'Actually failed' is judged exactly like evaluate.py judges an accepted
# document: any of the 10 FIELDS that differs from the labels after norm()
# (case/whitespace-insensitive; missing prediction = mismatch) is an error.
# The cell also builds cascade answers.jsonl and scores it with evaluate.py
# itself, so the reported numbers are consistent with the official metric.

import csv
import tempfile
from datetime import datetime

sys.path.insert(0, BASE_PATH)
from evaluate import FIELDS as EVAL_FIELDS, norm as eval_norm, main as eval_main


def eval_wrong_fields(gt_fields, viz_fields):
    """Fields evaluate.py would count as errors if this document were accepted."""
    wrong = []
    for f in EVAL_FIELDS:
        viz = viz_fields.get(f)
        pred = extract_predicted_text(viz) if viz is not None else None
        if eval_norm(pred) != eval_norm(gt_fields.get(f)):
            wrong.append(f)
    return wrong


def is_valid_date(text):
    """Exactly dd.mm.yyyy and a real calendar date."""
    if not re.fullmatch(r"\d{2}\.\d{2}\.\d{4}", text):
        return False
    try:
        datetime.strptime(text, "%d.%m.%Y")
        return True
    except ValueError:
        return False


def format_wrong_fields(pred_fields):
    """Predicted fields that violate their expected format (rule F)."""
    wrong = []
    for f, v in pred_fields.items():
        if v is None:
            continue
        v = eval_norm(v)
        if f in FMT_NAME_FIELDS:
            bad = re.search(r"\d", v) is not None
        elif f == "Sex_ru":
            bad = v not in ("МУЖ.", "ЖЕН.")
        elif f in FMT_DATE_FIELDS:
            bad = not is_valid_date(v)
        elif f == "Issue_organisation_code":
            bad = re.fullmatch(r"[\d-]*", v) is None
        elif f == "Licence_number":
            bad = re.fullmatch(r"[\d\s]*", v) is None
        elif f in FMT_TEXT_FIELDS:
            bad = FMT_TEXT_RE.fullmatch(v) is None
        else:
            bad = False
        if bad:
            wrong.append(f)
    return wrong


CHAR_CONF_T = 0.92
GLARE_MAX = 0.15
SHARPNESS_MIN = 125

# Field-format rules (F): charset/shape each predicted field must have.
FMT_NAME_FIELDS = ("Last_name_ru", "First_name_ru", "Middle_name_ru")
FMT_DATE_FIELDS = ("Birth_date", "Issue_date")
FMT_TEXT_FIELDS = ("Birth_place_ru", "Issue_organization_ru")
FMT_TEXT_RE = re.compile(r"[A-Za-zА-ЯЁа-яё\s.\d,()№/'\"\u2019-]*\Z")

SUBDIV_PATH = os.path.join(BASE_PATH, "reference/subdivisions.csv")
codes_ref, aliases_ref = set(), {}
with open(SUBDIV_PATH, encoding="utf-8") as f:
    for row in csv.DictReader(f):
        code = row["code"].strip()
        codes_ref.add(code)
        aliases_ref.setdefault(code, []).append(normalize(row["name"]))

RULES = ("R1 unknown code", "C  char conf", "M  missing field", "Q  quality gate", "F  format")
rule_flags = [0] * 6
rule_only = [0] * 6          # lines flagged ONLY by this rule
rule_hit = [0] * 6           # flagged by rule AND actually failed
field_errors = {f: 0 for f in EVAL_FIELDS}   # per-field errors, as in evaluate.py
fmt_errors = {f: 0 for f in EVAL_FIELDS}     # per-field format violations (rule F)
casc_tp = casc_fp = casc_fn = casc_tn = 0
n_lines = 0
answers = []                 # cascade decisions, scored by evaluate.py below

with open(LABELS_PATH, encoding="utf-8") as f_labels, open(PREDS_PATH, encoding="utf-8") as f_preds:
    for labels_line, preds_line in zip(f_labels, f_preds):
        label_rec, pred_rec = json.loads(labels_line), json.loads(preds_line)
        gt_fields, viz_fields = label_rec["fields"], pred_rec["viz"]
        n_lines += 1
        wrong_fields = eval_wrong_fields(gt_fields, viz_fields)
        actual_failed = bool(wrong_fields)   # evaluate.py: accepted with error
        for f in wrong_fields:
            field_errors[f] += 1

        # predicted values for this document (rule F operates on these)
        pred_fields = {f: extract_predicted_text(viz_fields[f]) if viz_fields.get(f) is not None else None
                       for f in EVAL_FIELDS}
        f_wrong = format_wrong_fields(pred_fields)
        for f in f_wrong:
            fmt_errors[f] += 1

        # reference rules
        code_viz = viz_fields.get("Issue_organisation_code")
        name_viz = viz_fields.get("Issue_organization_ru")
        pred_code = normalize(extract_predicted_text(code_viz)) if code_viz else None
        pred_name = normalize(extract_predicted_text(name_viz)) if name_viz else None
        r1 = pred_code is not None and pred_code not in codes_ref
        best_sim = 0.0
        if pred_code in aliases_ref and pred_name is not None:
            for ref_name in aliases_ref[pred_code]:
                best_sim = max(best_sim, SequenceMatcher(None, pred_name, ref_name).ratio())

        # char-conf rule + missing-field rule
        min_conf = None
        for field in EVAL_FIELDS:
            viz = viz_fields.get(field)
            if viz is None:
                continue
            _, confs = pred_string_with_conf(viz)
            for c in confs:
                if c is not None and (min_conf is None or c < min_conf):
                    min_conf = c
        m = any(viz_fields.get(f) is None for f in EVAL_FIELDS)
        c_flag = min_conf is not None and min_conf < CHAR_CONF_T

        # quality gate
        q = pred_rec.get("quality", {})
        q_flag = (q.get("glare", 0) > GLARE_MAX) or (q.get("sharpness", 10**9) < SHARPNESS_MIN)

        # field-format rule
        f_flag = bool(f_wrong)

        flags = [r1, c_flag, m, q_flag, f_flag]
        n_flagged = sum(flags)
        decision = "reject" if n_flagged else "accept"
        rec = {"doc_id": pred_rec["doc_id"], "decision": decision}
        if decision == "accept":
            rec["fields"] = {f: v for f, v in pred_fields.items() if v is not None}
        answers.append(rec)
        for i, flag in enumerate(flags):
            if flag:
                rule_flags[i] += 1
                rule_hit[i] += actual_failed
                if n_flagged == 1:
                    rule_only[i] += 1

        if n_flagged and actual_failed:
            casc_tp += 1
        elif n_flagged and not actual_failed:
            casc_fp += 1
        elif not n_flagged and actual_failed:
            casc_fn += 1
        else:
            casc_tn += 1

print(f"cascade rules: char_conf < {CHAR_CONF_T}, unknown code R1"
      f" missing field, glare > {GLARE_MAX} or sharpness < {SHARPNESS_MIN},"
      f" field format F")
print()
print("Per-rule contribution:")
print(f"{'rule':<18}{'flagged':>9}{'of them failed':>16}{'only rule':>11}")
for i, name in enumerate(RULES):
    print(f"{name:<18}{rule_flags[i]:>9}{rule_hit[i]:>16}{rule_only[i]:>11}")
print()

print("Confusion matrix (FULL cascade):")
print(f"{'':>18}{'actually FAILED':>18}{'actually OK':>14}{'total':>8}")
print(f"{'flagged bad':>18}{casc_tp:>18}{casc_fp:>14}{casc_tp + casc_fp:>8}")
print(f"{'approved':>18}{casc_fn:>18}{casc_tn:>14}{casc_fn + casc_tn:>8}")
print(f"{'total':>18}{casc_tp + casc_fn:>18}{casc_fp + casc_tn:>14}{n_lines:>8}")
print()
print(f"   caught fails (recall): {casc_tp}/{casc_tp + casc_fn}"
      f"  ({100 * casc_tp / (casc_tp + casc_fn):.1f}%)")
print(f"   false alarms: {casc_fp} of {casc_fp + casc_tn} OK lines"
      f"  ({100 * casc_fp / (casc_fp + casc_tn):.1f}%)")
print(f"   approved-but-failed (dangerous): {casc_fn}/{n_lines}"
      f"  ({100 * casc_fn / n_lines:.2f}% of all lines)")
print()

# Score the cascade answers with the official evaluate.py itself:
# 'принято верно' must equal casc_tn, 'принято с ошибкой' must equal casc_fn,
# and 'ошибки по полям' must match field_errors accumulated above.
answers_path = os.path.join(tempfile.mkdtemp(), "answers.jsonl")
with open(answers_path, "w", encoding="utf-8") as f:
    for rec in answers:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print("Official evaluate.py on cascade answers (train):")
eval_main(answers_path, LABELS_PATH)

print("Format violations per field (rule F):")
print("   " + ", ".join(f"{f}={c}" for f, c in fmt_errors.items() if c))

cascade rules: char_conf < 0.92, unknown code R1 missing field, glare > 0.15 or sharpness < 125, field format F

Per-rule contribution:
rule                flagged  of them failed  only rule
R1 unknown code         107              78         34
C  char conf           1114            1043        607
M  missing field        332             263        186
Q  quality gate         365             328         29
F  format               162             162         20

Confusion matrix (FULL cascade):
                     actually FAILED   actually OK   total
       flagged bad              1213           179    1392
          approved               107          1501    1608
             total              1320          1680    3000

   caught fails (recall): 1213/1320  (91.9%)
   false alarms: 179 of 1680 OK lines  (10.7%)
   approved-but-failed (dangerous): 107/3000  (3.57% of all lines)

Official evaluate.py on cascade answers (train):
документов:              3000
принято верно:          

In [9]:
# MRZ structure: what the mrz field contains and how reliable it is.
# Uses solution/mrzlib.py: layout, ICAO 7-3-1 check digits and the
# Cyrillic->Latin transliteration table, all derived on train (see module
# docstring). Line 2 (44 chars):
#   [0:9) doc number = licence series[:3]+number, [9] its check digit (CD),
#   [13:19) birth yymmdd, [19] CD, [20] sex M/F, [29:35) issue date yymmdd,
#   [35:41) org code 6 digits, [28:42) 'personal number' field, [42] CD,
#   [43] composite CD over [0:10)+[13:20)+[21:43).
# Check digits verify the MRZ itself WITHOUT ground truth -> free integrity
# signal (cd_fail_count) and a trust gate for cross-check rules.

import sys
sys.path.insert(0, os.getcwd())
import mrzlib

stats = Counter()
lic_rel = Counter()
n_docs = 0
with open(LABELS_PATH, encoding="utf-8") as f_labels, open(PREDS_PATH, encoding="utf-8") as f_preds:
    for labels_line, preds_line in zip(f_labels, f_preds):
        label_rec, pred_rec = json.loads(labels_line), json.loads(preds_line)
        n_docs += 1
        lines = ((pred_rec.get("mrz") or {}).get("lines", []))
        if len(lines) != 2:
            stats["no/short mrz"] += 1
            continue
        l2 = mrzlib.parse_line2(lines[1].get("text", ""))
        l1 = mrzlib.parse_line1(lines[0].get("text", ""))
        if l2 is None:
            stats["line2 parse fail"] += 1
            continue
        stats["line2 parsed"] += 1
        gt = label_rec["fields"]
        for ok_key, bad_key, good in (
            ("docnum CD ok", "docnum CD bad", l2["doc_num_cd_ok"]),
            ("birth CD ok", "birth CD bad", l2["birth_cd_ok"]),
            ("personal CD ok", "personal CD bad", l2["personal_cd_ok"]),
            ("composite CD ok", "composite CD bad", l2["composite_cd_ok"]),
        ):
            stats[ok_key if good else bad_key] += 1
        lic = re.sub(r"\D", "", gt["Licence_number"])
        lic_rel["num == lic[:3]+lic[4:]"] += l2["doc_num"] == lic[:3] + lic[4:]
        lic_rel["num != lic[:3]+lic[4:]"] += l2["doc_num"] != lic[:3] + lic[4:]
        want_bd = gt["Birth_date"][8:] + gt["Birth_date"][3:5] + gt["Birth_date"][:2]
        stats["birth == GT" if l2["birth"] == want_bd else "birth != GT"] += 1
        gt_sex = "M" if gt["Sex_ru"].startswith("МУЖ") else "F"
        stats["sex == GT" if l2["sex"] == gt_sex else "sex != GT"] += 1
        want_iss = gt["Issue_date"][8:] + gt["Issue_date"][3:5] + gt["Issue_date"][:2]
        stats["issue == GT" if l2["issue_date"] == want_iss else "issue != GT"] += 1
        code = gt["Issue_organisation_code"].replace("-", "")
        stats["code == GT" if l2["org_code"] == code else "code != GT"] += 1
        if l1:
            fio = "".join(gt[f] for f in ("Last_name_ru", "First_name_ru", "Middle_name_ru"))
            expected = mrzlib.translit(fio.replace(" ", ""))
            stats["translit == mrz names" if expected == l1["joined"] else "translit != mrz names"] += 1

print(f"docs: {n_docs}")
for k in sorted(stats):
    print(f"   {k:<26}{stats[k]:>6}")
print("   " + ", ".join(f"{k}={v}" for k, v in lic_rel.items()))


docs: 3000
   birth != GT                    7
   birth == GT                 2642
   birth CD bad                   8
   birth CD ok                 2641
   code != GT                    36
   code == GT                  2613
   composite CD bad              89
   composite CD ok             2560
   docnum CD bad                 10
   docnum CD ok                2639
   issue != GT                   34
   issue == GT                 2615
   line2 parse fail             109
   line2 parsed                2649
   no/short mrz                 242
   personal CD bad               65
   personal CD ok              2584
   sex == GT                   2649
   translit != mrz names        302
   translit == mrz names       2346
   num == lic[:3]+lic[4:]=2640, num != lic[:3]+lic[4:]=9


In [10]:
# Cross-check rules: visual zone vs MRZ disagreement => one of the two is wrong.
#   X1 birth date mismatch (MRZ [13:19) vs viz Birth_date)
#   X2 sex mismatch                        (shown below: never fires -> dropped)
#   X3 licence number mismatch             (MRZ doc number vs viz Licence_number)
#   X4 issue date / org code mismatch      (MRZ [29:35) / [35:41) vs viz)
#   X5 FIO translit similarity < T         (threshold sweep below)
#   X6 mrz cd_fail_count >= 2              (MRZ self-integrity)
# Key finding: viz and MRZ errors are PARTIALLY CORRELATED (the same corrupted
# source string seems rendered into both zones), so MRZ agreement does NOT
# prove a field is correct; only disagreement is actionable.

from datetime import datetime

REF_DATE = datetime(2026, 10, 1)
X_RULES = ("X1 birth", "X2 sex", "X3 licence", "X4 issue/code", "X6 mrz CD")
x_stat = {r: [0, 0] for r in X_RULES}      # [flagged_and_failed, flagged_and_ok]
x5_sweep = {t: [0, 0] for t in (0.85, 0.90, 0.92, 0.95, 0.97)}
fio_ok_scores, fio_bad_scores = [], []     # fio_score when FIO fields right/wrong
v_flags = Counter()                        # conservative validity rules
FIO_FIELDS = ("Last_name_ru", "First_name_ru", "Middle_name_ru")

with open(LABELS_PATH, encoding="utf-8") as f_labels, open(PREDS_PATH, encoding="utf-8") as f_preds:
    for labels_line, preds_line in zip(f_labels, f_preds):
        label_rec, pred_rec = json.loads(labels_line), json.loads(preds_line)
        gt_fields, viz_fields = label_rec["fields"], pred_rec["viz"]
        actual_failed = bool(eval_wrong_fields(gt_fields, viz_fields))
        cc = mrzlib.cross_check(viz_fields, pred_rec.get("mrz"))

        rule_hits = {
            "X1 birth": cc["birth_mismatch"] is True,
            "X2 sex": cc["sex_mismatch"] is True,
            "X3 licence": cc["licence_mismatch"] is True,
            "X4 issue/code": (cc["issue_date_mismatch"] is True) or (cc["org_code_mismatch"] is True),
            "X6 mrz CD": (cc["cd_fail_count"] or 0) >= 2,
        }
        for r, hit in rule_hits.items():
            if hit:
                x_stat[r][1 if actual_failed else 0] += 1
        if cc["fio_score"] is not None:
            for t in x5_sweep:
                if cc["fio_score"] < t:
                    x5_sweep[t][1 if actual_failed else 0] += 1
            fio_wrong = any(viz_fields.get(f) is None or
                            eval_norm(extract_predicted_text(viz_fields[f])) != eval_norm(gt_fields.get(f))
                            for f in FIO_FIELDS)
            (fio_bad_scores if fio_wrong else fio_ok_scores).append(cc["fio_score"])

        # conservative validity rules on PREDICTED dates (catch corrupted dates;
        # train GT never violates them, so on test they mostly catch OCR errors)
        bd_txt = extract_predicted_text(viz_fields['Birth_date']) if viz_fields.get('Birth_date') is not None else None
        iss_txt = extract_predicted_text(viz_fields['Issue_date']) if viz_fields.get('Issue_date') is not None else None
        try:
            b = datetime.strptime(bd_txt, '%d.%m.%Y')
            i = datetime.strptime(iss_txt, '%d.%m.%Y')
            v_flags["V1 issue>01.10.2026" if i > REF_DATE else "V1 ok"] += 1
            v_flags["V2 age<14 at issue" if (i - b).days / 365.25 < 14 else "V2 ok"] += 1
            v_flags["V3 issue<birth" if i < b else "V3 ok"] += 1
        except (ValueError, TypeError):
            v_flags["unparseable dates"] += 1

print("Per-rule standalone contribution (flagged = disagree with MRZ):")
print(f"{'rule':<16}{"flagged":>9}{"of them failed":>16}{"of them ok":>12}")
for r in X_RULES:
    print(f"{r:<16}{x_stat[r][0] + x_stat[r][1]:>9}{x_stat[r][0]:>16}{x_stat[r][1]:>12}")
print()
print("X5 (fio_score < T) sweep:  flagged_failed / flagged_ok")
for t, (ff, fo) in sorted(x5_sweep.items()):
    print(f"   T={t:<6}{ff:>5} / {fo}")
print()
for name, arr in (("fio_score, FIO fields correct", fio_ok_scores),
                  ("fio_score, FIO fields wrong  ", fio_bad_scores)):
    s = sorted(arr)
    if s:
        print(f"{name}: n={len(s)}  min={s[0]:.3f}  p5={s[len(s)//20]:.3f}  median={s[len(s)//2]:.3f}")
corr = sum(1 for s in fio_bad_scores if s == 1.0)
print(f"correlated errors: {corr} of {len(fio_bad_scores)} docs with wrong FIO "
      f"have fio_score == 1.0 (MRZ shows the SAME corrupted name)")
print()
print("Validity rules on predicted dates:", dict(v_flags))


Per-rule standalone contribution (flagged = disagree with MRZ):
rule              flagged  of them failed  of them ok
X1 birth               43               0          43
X2 sex                  0               0           0
X3 licence             70               4          66
X4 issue/code         130              20         110
X6 mrz CD              82              25          57

X5 (fio_score < T) sweep:  flagged_failed / flagged_ok
   T=0.85      0 / 43
   T=0.9       1 / 66
   T=0.92      1 / 78
   T=0.95      4 / 117
   T=0.97     47 / 270

fio_score, FIO fields correct: n=2276  min=0.873  p5=0.977  median=1.000
fio_score, FIO fields wrong  : n=372  min=0.629  p5=0.766  median=0.963
correlated errors: 106 of 372 docs with wrong FIO have fio_score == 1.0 (MRZ shows the SAME corrupted name)

Validity rules on predicted dates: {'V1 ok': 2835, 'V2 ok': 2844, 'V3 ok': 2844, 'unparseable dates': 148, 'V2 age<14 at issue': 8, 'V3 issue<birth': 8, 'V1 issue>01.10.2026': 17}


In [11]:
# Cascade v2: old rules (C char-conf, M missing, Q quality, F format, R1 code)
# + MRZ cross-checks + conservative validity rules.
# MRZ rules are gated on MRZ integrity (cd_fail_count == 0): a corrupted MRZ
# must not trigger rejections of a correct visual zone.
#   X1 birth mismatch            (43 TP / 0 FP on train, ungated)
#   X3 licence mismatch, gated   (56 TP / 0 FP)
#   X4a issue date mismatch, gated (34 TP / 0 FP)
#   X4b org code mismatch, gated   (35 TP / 1 FP)
#   V1 predicted issue date > 01.10.2026
#   V2 predicted age at issue < 14
#   V3 predicted issue date < birth date
# Not used as standalone rules: X2 sex (never fires), X5 FIO similarity
# (0 dangerous catches among approved docs at T=0.92; kept as a feature for
# the OCR-error classifier), X6 CD-fail (19 FP per 1 TP), and the 'approve
# old-flagged docs on MRZ agreement' override (rejected empirically: it
# approves docs whose Birth_place/Issue_org errors the MRZ cannot see,
# 517 accepted-with-error).

m_rz_stat = {r: [0, 0, 0] for r in ("old", "X1", "X3", "X4a", "X4b", "V1", "V2", "V3")}
# [flagged, of them failed, flagged ONLY by this rule group]
casc = {"tp": 0, "fp": 0, "fn": 0, "tn": 0}
answers = []

with open(LABELS_PATH, encoding="utf-8") as f_labels, open(PREDS_PATH, encoding="utf-8") as f_preds:
    for labels_line, preds_line in zip(f_labels, f_preds):
        label_rec, pred_rec = json.loads(labels_line), json.loads(preds_line)
        gt_fields, viz_fields = label_rec["fields"], pred_rec["viz"]
        wrong_fields = eval_wrong_fields(gt_fields, viz_fields)
        actual_failed = bool(wrong_fields)
        pred_fields = {f: extract_predicted_text(viz_fields[f]) if viz_fields.get(f) is not None else None
                       for f in EVAL_FIELDS}
        f_wrong = format_wrong_fields(pred_fields)

        pred_code = normalize(pred_fields["Issue_organisation_code"]) if pred_fields["Issue_organisation_code"] else None
        r1 = pred_code is not None and pred_code not in codes_ref
        min_conf = None
        for field in EVAL_FIELDS:
            viz = viz_fields.get(field)
            if viz is None:
                continue
            _, confs = pred_string_with_conf(viz)
            for c in confs:
                if c is not None and (min_conf is None or c < min_conf):
                    min_conf = c
        m_flag = any(viz_fields.get(f) is None for f in EVAL_FIELDS)
        c_flag = min_conf is not None and min_conf < CHAR_CONF_T
        q = pred_rec.get("quality", {})
        q_flag = (q.get("glare", 0) > GLARE_MAX) or (q.get("sharpness", 10**9) < SHARPNESS_MIN)
        f_flag = bool(f_wrong)
        old_any = r1 or c_flag or m_flag or q_flag or f_flag

        cc = mrzlib.cross_check(viz_fields, pred_rec.get("mrz"))
        clean_mrz = cc["mrz_ok"] and cc["cd_fail_count"] == 0
        x1 = cc["birth_mismatch"] is True
        x3 = clean_mrz and cc["licence_mismatch"] is True
        x4a = clean_mrz and cc["issue_date_mismatch"] is True
        x4b = clean_mrz and cc["org_code_mismatch"] is True

        v1 = v2 = v3 = False
        try:
            b = datetime.strptime(pred_fields["Birth_date"] or "", "%d.%m.%Y")
            i = datetime.strptime(pred_fields["Issue_date"] or "", "%d.%m.%Y")
            v1 = i > REF_DATE
            v2 = (i - b).days / 365.25 < 14
            v3 = i < b
        except (ValueError, TypeError):
            pass

        groups = {"old": old_any, "X1": x1, "X3": x3, "X4a": x4a, "X4b": x4b,
                  "V1": v1, "V2": v2, "V3": v3}
        reject = any(groups.values())
        for name, flag in groups.items():
            if flag:
                m_rz_stat[name][0] += 1
                m_rz_stat[name][1] += actual_failed
        newly = [n for n, fl in groups.items() if fl]
        if len(newly) == 1:
            m_rz_stat[newly[0]][2] += 1

        rec = {"doc_id": pred_rec["doc_id"], "decision": "reject" if reject else "accept"}
        if rec["decision"] == "accept":
            rec["fields"] = {f: v for f, v in pred_fields.items() if v is not None}
        answers.append(rec)
        if reject and actual_failed:
            casc["tp"] += 1
        elif reject and not actual_failed:
            casc["fp"] += 1
        elif actual_failed:
            casc["fn"] += 1
        else:
            casc["tn"] += 1

print("Per-group contribution:")
print(f"{'group':<8}{"flagged":>9}{"of them failed":>16}{"only flag":>11}")
for name in m_rz_stat:
    fl, hit, only = m_rz_stat[name]
    print(f"{name:<8}{fl:>9}{hit:>16}{only:>11}")
print()
print("Confusion matrix (cascade v2):")
print(f"{'':>18}{"actually FAILED":>18}{"actually OK":>14}{"total":>8}")
print(f"{'flagged bad':>18}{casc['tp']:>18}{casc['fp']:>14}{casc['tp'] + casc['fp']:>8}")
print(f"{'approved':>18}{casc['fn']:>18}{casc['tn']:>14}{casc['fn'] + casc['tn']:>8}")
print(f"{'total':>18}{casc['tp'] + casc['fn']:>18}{casc['fp'] + casc['tn']:>14}{sum(casc.values()):>8}")
print()
print(f"   caught fails (recall): {casc['tp']}/{casc['tp'] + casc['fn']}"
      f"  ({100 * casc['tp'] / max(1, casc['tp'] + casc['fn']):.1f}%)")
print(f"   false alarms: {casc['fp']} of {casc['fp'] + casc['tn']} OK lines"
      f"  ({100 * casc['fp'] / max(1, casc['fp'] + casc['tn']):.1f}%)")
print(f"   approved-but-failed (dangerous): {casc['fn']}/{sum(casc.values())}"
      f"  ({100 * casc['fn'] / sum(casc.values()):.2f}% of all lines)")
print()

answers_path = os.path.join(tempfile.mkdtemp(), "answers_mrz.jsonl")
with open(answers_path, "w", encoding="utf-8") as f:
    for rec in answers:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
print("Official evaluate.py on cascade v2 answers (train):")
eval_main(answers_path, LABELS_PATH)


Per-group contribution:
group     flagged  of them failed  only flag
old          1392            1213       1225
X1             43              43          3
X3             56              56          6
X4a            34              34          3
X4b            36              35          1
V1             17              17          0
V2              8               8          0
V3              8               8          0

Confusion matrix (cascade v2):
                     actually FAILED   actually OK   total
       flagged bad              1228           179    1407
          approved                92          1501    1593
             total              1320          1680    3000

   caught fails (recall): 1228/1320  (93.0%)
   false alarms: 179 of 1680 OK lines  (10.7%)
   approved-but-failed (dangerous): 92/3000  (3.07% of all lines)

Official evaluate.py on cascade v2 answers (train):
документов:              3000
принято верно:           1501  (50.0%)
принято с ошибкой:    

In [12]:
# Per-field OCR-error classifier (the 'separate model' idea).
# Task: for one text-field value predict P(it contains an OCR error).
# Labels are free on train: compare prediction vs GT (norm-insensitive).
# Only the 5 free-text fields (3x FIO, Birth_place_ru, Issue_organization_ru);
# dates/code/sex/licence are fully covered by deterministic rules.
# Features (errfeatures.py): char-conf shape, format plausibility, char
# trigram LM fitted on train GT of the same field (OOV-trigram rate catches
# 'СМИРНОВ'->'СМИРНСВ' even at char_conf 1.0), doc-level MRZ features,
# doc-level image quality.
# Lexicon-similarity features were tried and REMOVED: on train the value's
# own GT sits in the vocabulary (median sim 1.0), but on test values are
# unseen (median 0.67) -> the model equates 'no lexicon support' with 'error'
# and rejects 1092 of 1096 clean test docs. Without the feature the test
# p_max distribution is healthy (median 0.009).
# HONEST protocol: LM for a fold is fitted on train-fold GT only,
# CV is grouped by document. (Fitting on all train GT gives AUC 0.9998 —
# pure leakage; fold-exclusive gives the realistic estimate below.)

import sys
import time
from collections import defaultdict
sys.path.insert(0, os.getcwd())
import errfeatures
from errfeatures import TEXT_FIELDS

records = []
with open(LABELS_PATH, encoding="utf-8") as f_labels, open(PREDS_PATH, encoding="utf-8") as f_preds:
    for labels_line, preds_line in zip(f_labels, f_preds):
        records.append((json.loads(labels_line), json.loads(preds_line)))

extra_lexicon = {"Issue_organization_ru": []}
with open(SUBDIV_PATH, encoding="utf-8") as f:
    for row in csv.DictReader(f):
        extra_lexicon["Issue_organization_ru"].append(row["name"])

# pass 1: static features + labels
rows, y_rows, groups, texts, fields_ = [], [], [], [], []
doc_first_row = {}
for doc_i, (label_rec, pred_rec) in enumerate(records):
    gt_fields, viz_fields = label_rec["fields"], pred_rec["viz"]
    cc = mrzlib.cross_check(viz_fields, pred_rec.get("mrz"))
    mfeat = errfeatures.mrz_doc_features(cc)
    qfeat = errfeatures.quality_features(pred_rec)
    for field in TEXT_FIELDS:
        viz = viz_fields.get(field)
        if viz is None:
            continue  # already flagged by rule M
        doc_first_row.setdefault(doc_i, len(rows))
        text, confs, feats = errfeatures.char_conf_features(viz)
        feats.update(errfeatures.TextFeatureBuilder.format_features(field, text))
        feats.update(mfeat)
        feats.update(qfeat)
        rows.append(feats)
        texts.append(text)
        fields_.append(field)
        y_rows.append(0 if eval_norm(text) == eval_norm(gt_fields[field]) else 1)
        groups.append(doc_i)

import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score

n = len(rows)
y_arr = np.array(y_rows)
groups_arr = np.array(groups)
fold_of = np.zeros(n, dtype=int)
for k, (_, te) in enumerate(GroupKFold(n_splits=5).split(np.zeros(n), y_arr, groups_arr)):
    fold_of[te] = k
doc_fold = {doc_i: fold_of[r] for doc_i, r in doc_first_row.items()}

# pass 2: fold-exclusive LM + lexicon features
t0 = time.time()
for k in range(5):
    corpus = defaultdict(list)
    for doc_i, (label_rec, _) in enumerate(records):
        if doc_fold.get(doc_i) == k:
            continue
        for field in TEXT_FIELDS:
            corpus[field].append(label_rec["fields"][field])
    builder = errfeatures.TextFeatureBuilder().fit(corpus, extra_lexicon)
    for i in range(n):
        if fold_of[i] != k:
            continue
        rows[i].update(builder.lm_features(fields_[i], texts[i]))
print(f"fold-exclusive LM/lexicon features built in {time.time() - t0:.0f}s")

keys = sorted(set().union(*(r.keys() for r in rows)))
X = np.array([[r.get(k, 0.0) for k in keys] for r in rows], dtype=float)

oof = np.zeros(n)
for tr, te in GroupKFold(n_splits=5).split(X, y_arr, groups_arr):
    clf = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.1, random_state=0)
    clf.fit(X[tr], y_arr[tr])
    oof[te] = clf.predict_proba(X[te])[:, 1]

print(f"OOF AUC (all text fields): {roc_auc_score(y_arr, oof):.4f}")
for field in TEXT_FIELDS:
    sel = np.array([f == field for f in fields_])
    if sel.sum() > 100 and 0 < y_arr[sel].sum() < sel.sum():
        print(f"   {field:<26} n={sel.sum():<5} pos={int(y_arr[sel].sum()):<4} AUC={roc_auc_score(y_arr[sel], oof[sel]):.4f}")
print("field-level threshold sweep (flag field if p > T):")
for t in (0.3, 0.5, 0.7, 0.8, 0.9):
    flag = oof > t
    print(f"   T={t:<5} wrong_caught={int((flag & (y_arr == 1)).sum()):<5}"
          f" ok_flagged={int((flag & (y_arr == 0)).sum())}")

# doc-level score for the cascade: max p over the text fields of the doc
doc_p = defaultdict(float)
for p, g in zip(oof, groups):
    doc_p[g] = max(doc_p[g], p)


fold-exclusive LM/lexicon features built in 2s
OOF AUC (all text fields): 0.9772
   Last_name_ru               n=2977  pos=113  AUC=0.9469
   First_name_ru              n=2981  pos=77   AUC=0.9299
   Middle_name_ru             n=2859  pos=119  AUC=0.9755
   Birth_place_ru             n=2981  pos=500  AUC=0.9861
   Issue_organization_ru      n=2973  pos=575  AUC=0.9768
field-level threshold sweep (flag field if p > T):
   T=0.3   wrong_caught=1189  ok_flagged=124
   T=0.5   wrong_caught=1150  ok_flagged=67
   T=0.7   wrong_caught=1112  ok_flagged=29
   T=0.8   wrong_caught=1090  ok_flagged=11
   T=0.9   wrong_caught=1047  ok_flagged=3


In [13]:
# Cascade v3 = cascade v2 + rule S: reject the doc if the OCR-error classifier
# gives p > T for any text field (out-of-fold probabilities from the previous
# cell -> an honest estimate of test-time behaviour).

ML_T_SWEEP = (0.5, 0.7, 0.9)

for ML_T in ML_T_SWEEP:
    casc = {"tp": 0, "fp": 0, "fn": 0, "tn": 0}
    answers = []
    with open(LABELS_PATH, encoding="utf-8") as f_labels, open(PREDS_PATH, encoding="utf-8") as f_preds:
        for doc_i, (labels_line, preds_line) in enumerate(zip(f_labels, f_preds)):
            label_rec, pred_rec = json.loads(labels_line), json.loads(preds_line)
            gt_fields, viz_fields = label_rec["fields"], pred_rec["viz"]
            actual_failed = bool(eval_wrong_fields(gt_fields, viz_fields))
            pred_fields = {f: extract_predicted_text(viz_fields[f]) if viz_fields.get(f) is not None else None
                           for f in EVAL_FIELDS}
            f_wrong = format_wrong_fields(pred_fields)

            pred_code = normalize(pred_fields["Issue_organisation_code"]) if pred_fields["Issue_organisation_code"] else None
            r1 = pred_code is not None and pred_code not in codes_ref
            min_conf = None
            for field in EVAL_FIELDS:
                viz = viz_fields.get(field)
                if viz is None:
                    continue
                _, confs = pred_string_with_conf(viz)
                for c in confs:
                    if c is not None and (min_conf is None or c < min_conf):
                        min_conf = c
            m_flag = any(viz_fields.get(f) is None for f in EVAL_FIELDS)
            c_flag = min_conf is not None and min_conf < CHAR_CONF_T
            q = pred_rec.get("quality", {})
            q_flag = (q.get("glare", 0) > GLARE_MAX) or (q.get("sharpness", 10**9) < SHARPNESS_MIN)
            f_flag = bool(f_wrong)

            cc = mrzlib.cross_check(viz_fields, pred_rec.get("mrz"))
            clean_mrz = cc["mrz_ok"] and cc["cd_fail_count"] == 0
            mrz_flag = (cc["birth_mismatch"] is True
                        or (clean_mrz and cc["licence_mismatch"] is True)
                        or (clean_mrz and cc["issue_date_mismatch"] is True)
                        or (clean_mrz and cc["org_code_mismatch"] is True))

            val_flag = False
            try:
                b = datetime.strptime(pred_fields["Birth_date"], "%d.%m.%Y")
                i = datetime.strptime(pred_fields["Issue_date"], "%d.%m.%Y")
                val_flag = i > REF_DATE or (i - b).days / 365.25 < 14 or i < b
            except (ValueError, TypeError):
                pass

            s_flag = doc_p.get(doc_i, 0.0) > ML_T
            reject = r1 or c_flag or m_flag or q_flag or f_flag or mrz_flag or val_flag or s_flag

            rec = {"doc_id": pred_rec["doc_id"], "decision": "reject" if reject else "accept"}
            if rec["decision"] == "accept":
                rec["fields"] = {f: v for f, v in pred_fields.items() if v is not None}
            answers.append(rec)
            if reject and actual_failed:
                casc["tp"] += 1
            elif reject and not actual_failed:
                casc["fp"] += 1
            elif actual_failed:
                casc["fn"] += 1
            else:
                casc["tn"] += 1

    print(f"===== S threshold = {ML_T} =====")
    print(f"   flagged bad {casc['tp'] + casc['fp']}, approved {casc['fn'] + casc['tn']},"
          f" approved-but-failed = {casc['fn']}")
    answers_path = os.path.join(tempfile.mkdtemp(), f"answers_ml_{ML_T}.jsonl")
    with open(answers_path, "w", encoding="utf-8") as f:
        for rec in answers:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    print("   Official evaluate.py:")
    eval_main(answers_path, LABELS_PATH)
    print()


===== S threshold = 0.5 =====
   flagged bad 1420, approved 1580, approved-but-failed = 81
   Official evaluate.py:
документов:              3000
принято верно:           1499  (50.0%)
принято с ошибкой:       81
ошибки по полям:         Last_name_ru=17, First_name_ru=8, Middle_name_ru=9, Birth_date=2, Birth_place_ru=20, Licence_number=8, Issue_organization_ru=18

===== S threshold = 0.7 =====
   flagged bad 1417, approved 1583, approved-but-failed = 82
   Official evaluate.py:
документов:              3000
принято верно:           1501  (50.0%)
принято с ошибкой:       82
ошибки по полям:         Last_name_ru=17, First_name_ru=8, Middle_name_ru=9, Birth_date=2, Birth_place_ru=21, Licence_number=8, Issue_organization_ru=18

===== S threshold = 0.9 =====
   flagged bad 1414, approved 1586, approved-but-failed = 85
   Official evaluate.py:
документов:              3000
принято верно:           1501  (50.0%)
принято с ошибкой:       85
ошибки по полям:         Last_name_ru=17, First_name_